# SurvFace 06. Step 1 압축 특성 분석

SurvFace `training_set` development에서 ArcFace origin-512 압축기를 fit하고 training calibration에서 threshold를 선택한 뒤, 공식 gallery/mated/unmated test에서 post-hoc PCA/PQ 영향을 측정합니다. 공식 test는 fit 또는 calibration에 사용하지 않으며 원본 재탐색 fallback도 수행하지 않습니다.

성공 기준:

- training development와 calibration identity가 분리되고 official test가 두 split에 포함되지 않습니다.
- PCA-384/256/128/64/32와 PQ가 모두 origin-512에서 독립적으로 시작합니다.
- 공식 test의 순서와 mated/unmated 역할을 유지합니다.
- frozen origin threshold와 profile별 recalibrated threshold 결과를 함께 기록합니다.
- 모든 결과에서 `origin_fallback_used=False`를 검증합니다.


In [1]:
from __future__ import annotations

# 실행 범위: 이 셀의 세 값만 바꾸고 Kernel Restart -> Run All
MODE = "real"             # "dev" 또는 "real"
DATA_FRACTION = 1.0      # 0 < DATA_FRACTION <= 1
SEED = 42

MODEL_NAME = "arcface"
EXECUTE_STAGE = True
WRITE_OUTPUTS = True
OVERWRITE = True          # canonical stage result replacement
TARGET_FPIR = 0.10
TOP_K = 20
CALIBRATION_GALLERY_IDENTITIES = 100

import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
from IPython.display import display

def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "research").is_dir() and (candidate / "configs").is_dir():
            return candidate
    raise FileNotFoundError("D:/ronbun 내부에서 노트북을 실행하십시오.")

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

from research.calibration.rejection import choose_threshold
from research.compression import (
    ORIGIN_512,
    ORIGIN_EMBEDDING_DIMENSION,
    PCA_SWEEP_DIMENSIONS,
    pq_profile_name,
)
from research.database import create_database_engine, load_database_settings
from research.evaluation import (
    PAIRED_EMBEDDING_COLUMNS,
    RETRIEVAL_COMPARISON_COLUMNS,
    apply_retrieval_thresholds,
    compare_cosine_retrieval,
    paired_embedding_metrics,
)
from research.experiments import load_survface_compressor_bundle
from research.experiments.lfw_certification import fetch_lfw_vector_records
from research.experiments.scope import ExperimentScope
from research.protocols import (
    build_calibration_protocol,
    build_survface_official_protocol,
    validate_identity_disjoint_splits,
)
from research.runtime import ProgressReporter, RunStore, resolve_active_run
from research.runtime.hashing import sha256_file
from research.templates.aggregation import aggregate_templates

PCA_DIMENSIONS = (384, 256, 128, 64, 32)
EXPECTED_PQ_M = (8, 16, 32, 64, 128)
PQ_SOURCE_DIMENSION = 512
if PCA_DIMENSIONS != tuple(PCA_SWEEP_DIMENSIONS):
    raise RuntimeError("노트북 PCA sweep과 research.compression 정의가 다릅니다.")
if PQ_SOURCE_DIMENSION != ORIGIN_EMBEDDING_DIMENSION:
    raise RuntimeError("PQ는 origin-512에 직접 적용해야 합니다.")

EXPERIMENT_SCOPE = ExperimentScope(
    mode=MODE, data_fraction=DATA_FRACTION, seed=SEED
)
CONFIG_PATH = PROJECT_ROOT / "configs" / "experiments" / "step1_embedding_compression.yaml"
DATA_DIR = PROJECT_ROOT / "data" / "interim" / "survface"
TRAINING_MANIFEST_PATH = DATA_DIR / "training_manifest.csv"
TRAINING_SUMMARY_PATH = DATA_DIR / "training_summary.json"
OFFICIAL_MANIFEST_PATH = DATA_DIR / "official_manifest.csv"
OFFICIAL_SUMMARY_PATH = DATA_DIR / "summary.json"
RESULT_ROOT = PROJECT_ROOT / "results" / "step1" / "survface"
RUN_ROOT = PROJECT_ROOT / "runs" / "survface"
PROGRESS = ProgressReporter(
    "SurvFace Step 1 characterization",
    heartbeat_seconds=None,
    milestone_percent=10,
)

config = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8")) or {}
config.setdefault("execution", {}).update(EXPERIMENT_SCOPE.as_dict())
dataset_config = config["datasets"]["survface"]
pca_config = config["compression"]["families"]["pca"]
pq_config = config["compression"]["families"]["pq"]
PQ_SETTINGS = tuple(
    (int(setting["m"]), int(setting["nbits"]))
    for setting in pq_config["settings"]
)
if not PQ_SETTINGS or len(set(PQ_SETTINGS)) != len(PQ_SETTINGS):
    raise ValueError("Step 1 PQ settings must be non-empty and unique.")
if tuple(m for m, _ in PQ_SETTINGS) != EXPECTED_PQ_M:
    raise ValueError(f"Step 1 PQ m values must be {EXPECTED_PQ_M}, got {PQ_SETTINGS}.")
if any(nbits != 8 for _, nbits in PQ_SETTINGS):
    raise ValueError("Step 1 PQ settings currently require nbits=8.")
if any(PQ_SOURCE_DIMENSION % m != 0 for m, _ in PQ_SETTINGS):
    raise ValueError("Every PQ m must divide the origin-512 dimension.")
configured_training_manifest = dataset_config.get("training_manifest_path")
configured_official_manifest = dataset_config.get("manifest_path")
if not configured_training_manifest or (PROJECT_ROOT / configured_training_manifest).resolve() != TRAINING_MANIFEST_PATH.resolve():
    raise ValueError("Step 1 config의 SurvFace training_manifest_path가 노트북 경로와 다릅니다.")
if not configured_official_manifest or (PROJECT_ROOT / configured_official_manifest).resolve() != OFFICIAL_MANIFEST_PATH.resolve():
    raise ValueError("Step 1 config의 SurvFace manifest_path가 노트북 경로와 다릅니다.")
if dataset_config.get("protocol_adapter") != "survface_official":
    raise ValueError("SurvFace protocol_adapter는 official이어야 합니다.")
if dataset_config.get("official_manifest_role") != "evaluation_only":
    raise ValueError("SurvFace official manifest는 evaluation_only여야 합니다.")
if dataset_config.get("fit_source") != "same_dataset_training_development":
    raise ValueError("SurvFace fit source는 training development여야 합니다.")
if config["compression"].get("fit_split") != "development":
    raise ValueError("Step 1 compressor fit_split은 development여야 합니다.")
if config["models"].get("enabled") != [MODEL_NAME]:
    raise ValueError("현재 Step 1 실행 모델은 ArcFace 하나여야 합니다.")
if tuple(int(value) for value in pca_config["dimensions"]) != PCA_DIMENSIONS:
    raise ValueError("Step 1 config의 PCA dimensions가 노트북과 다릅니다.")
if int(pq_config["source_dimension"]) != PQ_SOURCE_DIMENSION:
    raise ValueError("Step 1 PQ source_dimension은 512여야 합니다.")
if bool(config["evaluation"].get("exact_fallback", True)):
    raise ValueError("Step 1 config에서 exact_fallback은 false여야 합니다.")

display(pd.Series({
    **EXPERIMENT_SCOPE.as_dict(),
    "model": MODEL_NAME,
    "execute_stage": EXECUTE_STAGE,
    "write_outputs": WRITE_OUTPUTS,
    "pca_dimensions": PCA_DIMENSIONS,
    "pq_settings": PQ_SETTINGS,
    "pq_source_dimension": PQ_SOURCE_DIMENSION,
    "target_fpir": TARGET_FPIR,
}, name="value").to_frame())


,value
mode,real
data_fraction,1.0
seed,42
is_full_dataset,True
is_paper_run,True
model,arcface
execute_stage,True
write_outputs,True
pca_dimensions,"(384, 256, 128, 64, 32)"
pq_settings,"((8, 8), (16, 8), (32, 8), (64, 8), (128, 8))"


## Plan

1. 선택된 training/official manifest와 extraction run의 origin-512 임베딩을 결합합니다.
2. Training development에서 PCA family와 direct-origin PQ를 fit합니다.
3. Training calibration open-set protocol에서 origin/profile threshold를 선택합니다.
4. 공식 SurvFace gallery/mated/unmated protocol에서 paired distortion과 exact cosine retrieval 변화를 계산합니다.
5. 데이터셋별 두 표와 요약 및 입력 hash를 별도 결과 디렉터리에 내보냅니다.


In [2]:
def resolve_extraction_run() -> tuple[Path | None, str | None]:
    try:
        path = resolve_active_run(
            RUN_ROOT,
            environment_variable="RONBUN_SURVFACE_RUN_DIR",
            allow_completed=True,
        )
        return path, None
    except Exception as exc:
        return None, f"{type(exc).__name__}: {exc}"

def validate_prepared_scope(summary_path: Path, *, label: str):
    if not summary_path.is_file():
        return None
    payload = json.loads(summary_path.read_text(encoding="utf-8"))
    prepared_scope = payload.get("scope")
    expected_scope = EXPERIMENT_SCOPE.as_dict()
    matches_expected = prepared_scope == expected_scope
    warning = None
    if not matches_expected:
        warning = (
            f"{label} scope mismatch. Re-run notebooks/survface/00_data_preparation/00_data_preparation.ipynb "
            f"with the current MODE/DATA_FRACTION/SEED. "
            f"expected={expected_scope}, prepared={prepared_scope}"
        )
        if EXECUTE_STAGE:
            raise ValueError(warning)
    return {"scope": prepared_scope, "matches_expected": matches_expected, "warning": warning}

required_paths = {
    "config": CONFIG_PATH,
    "training_manifest": TRAINING_MANIFEST_PATH,
    "training_summary": TRAINING_SUMMARY_PATH,
    "official_manifest": OFFICIAL_MANIFEST_PATH,
    "official_summary": OFFICIAL_SUMMARY_PATH,
}
RUN_DIR, RUN_RESOLUTION_ERROR = resolve_extraction_run()
preflight = {
    "scope": EXPERIMENT_SCOPE.as_dict(),
    "inputs_exist": {name: path.is_file() for name, path in required_paths.items()},
    "run_dir": str(RUN_DIR) if RUN_DIR else None,
    "run_resolution_error": RUN_RESOLUTION_ERROR,
    "execute_stage": EXECUTE_STAGE,
}
preflight["training_prepared_scope"] = validate_prepared_scope(
    TRAINING_SUMMARY_PATH, label="SurvFace training_summary.json"
)
preflight["official_prepared_scope"] = validate_prepared_scope(
    OFFICIAL_SUMMARY_PATH, label="SurvFace summary.json"
)

training_manifest = None
official_manifest = None
if TRAINING_MANIFEST_PATH.is_file():
    training_manifest = pd.read_csv(TRAINING_MANIFEST_PATH)
    required_training = {"image_id", "identity_id", "split", "image_path"}
    missing = sorted(required_training.difference(training_manifest.columns))
    if missing:
        raise ValueError(f"SurvFace training manifest missing columns: {missing}")
    if set(training_manifest["split"].astype(str)) != {"development", "calibration"}:
        raise ValueError("Training manifest는 development/calibration만 포함해야 합니다.")
    validate_identity_disjoint_splits(training_manifest)
    preflight["training_rows"] = int(len(training_manifest))
    display(training_manifest.groupby("split").agg(
        images=("image_id", "size"),
        identities=("identity_id", "nunique"),
    ))

if OFFICIAL_MANIFEST_PATH.is_file():
    official_manifest = pd.read_csv(OFFICIAL_MANIFEST_PATH)
    required_official = {
        "image_id", "identity_id", "split", "image_path",
        "protocol_role", "protocol_index",
    }
    missing = sorted(required_official.difference(official_manifest.columns))
    if missing:
        raise ValueError(f"SurvFace official manifest missing columns: {missing}")
    if not official_manifest["split"].astype(str).eq("test").all():
        raise ValueError("Official manifest는 test 행만 포함해야 합니다.")
    validate_identity_disjoint_splits(official_manifest)
    if training_manifest is not None and set(
        training_manifest["identity_id"].astype(str)
    ).intersection(official_manifest["identity_id"].astype(str)):
        raise ValueError("SurvFace training과 official test identity가 겹칩니다.")
    if training_manifest is not None:
        validate_identity_disjoint_splits(pd.concat(
            [training_manifest, official_manifest], ignore_index=True, sort=False
        ))
    preflight["official_rows"] = int(len(official_manifest))
    display(official_manifest.groupby("protocol_role", sort=False).size().rename("rows").to_frame())

display(pd.Series(preflight, name="value").to_frame())


,images,identities
split,,
calibration,44683,1064
development,176205,4255


,rows
protocol_role,
gallery,60294
registered_probe,60423
unknown_unknown_probe,121736


,value
scope,"{'mode': 'real', 'data_fraction': 1.0, 'seed':..."
inputs_exist,"{'config': True, 'training_manifest': True, 't..."
run_dir,C:\ronbun\runs\survface\2026\07\29\20260729-R0...
run_resolution_error,None
execute_stage,True
training_prepared_scope,"{'scope': {'data_fraction': 1.0, 'is_full_data..."
official_prepared_scope,"{'scope': {'data_fraction': 1.0, 'is_full_data..."
training_rows,220888
official_rows,242453


## Load origin embeddings

이 단계는 선택된 training과 공식 test 경로에 대응하는 `origin_512`만 DB에서 읽습니다. 기존 로더는 경로-벡터 결합에만 사용하고 certificate/fallback 경로는 호출하지 않습니다. 두 manifest의 모든 행에 512D 임베딩이 있어야 합니다.


In [3]:
study_rows = None
run = None
if EXECUTE_STAGE:
    missing_inputs = [name for name, exists in preflight["inputs_exist"].items() if not exists]
    if missing_inputs:
        raise FileNotFoundError(f"Missing Step 1 inputs: {missing_inputs}")
    if RUN_DIR is None:
        raise RuntimeError(f"SurvFace extraction run을 찾지 못했습니다: {RUN_RESOLUTION_ERROR}")

    run = RunStore.open(RUN_DIR)
    all_manifest = pd.concat(
        [training_manifest, official_manifest],
        ignore_index=True,
        sort=False,
    )
    if all_manifest["image_id"].duplicated().any():
        raise ValueError("Training/official 결합 image_id는 유일해야 합니다.")
    engine = create_database_engine(load_database_settings())
    vector_records = fetch_lfw_vector_records(
        engine,
        run_uid=run.run_id,
        image_paths=set(all_manifest["image_path"].astype(str)),
        project_root=PROJECT_ROOT,
        compression_profile=ORIGIN_512,
    )
    if vector_records.empty:
        raise ValueError("선택 manifest에 대응하는 origin-512 임베딩이 없습니다.")

    def canonical_path(value: str) -> str:
        path = Path(value)
        resolved = path if path.is_absolute() else PROJECT_ROOT / path
        return os.path.normcase(str(resolved.resolve()))

    keyed_manifest = all_manifest.copy()
    keyed_manifest["canonical_path"] = keyed_manifest["image_path"].map(canonical_path)
    study_rows = keyed_manifest.merge(
        vector_records[["canonical_path", "origin_embedding"]],
        on="canonical_path",
        how="left",
        validate="one_to_one",
    )
    missing_embeddings = study_rows["origin_embedding"].isna()
    if missing_embeddings.any():
        examples = study_rows.loc[missing_embeddings, "image_id"].astype(str).head(10).tolist()
        raise ValueError(
            f"origin-512 coverage incomplete: missing={int(missing_embeddings.sum())}, "
            f"examples={examples}"
        )
    dimensions = study_rows["origin_embedding"].map(
        lambda value: int(np.asarray(value).shape[0])
    )
    if not dimensions.eq(ORIGIN_EMBEDDING_DIMENSION).all():
        raise ValueError("모든 origin embedding은 512D여야 합니다.")
    display(study_rows.groupby(["split", "protocol_role"], dropna=False).size().rename("embedded_rows").to_frame())
else:
    print("EXECUTE_STAGE=False: DB를 읽거나 compressor를 fit하지 않았습니다.")


embedded_rows
split       protocol_role                       
calibration training                       44683
development training                      176205
test        gallery                        60294
            registered_probe               60423
            unknown_unknown_probe         121736

## Compression and fallback-free evaluation

Official gallery의 같은 identity 이미지들을 mean template로 집계한 뒤 template 자체를 압축합니다. Origin과 compressed cosine 검색은 각각 독립 실행하고, 압축 결과를 원본 검색 결과로 대체하지 않습니다. SurvFace official test score로 threshold를 다시 fit하지 않습니다. `frozen_origin`은 양쪽 표현에 origin training-calibration threshold를 적용하고, `recalibrated_compressed`는 origin에는 같은 origin threshold를 유지하면서 compressed에만 profile별 threshold를 적용합니다. `threshold_crossing`은 각 표현이 해당 정책의 threshold에서 내린 decision이 서로 다른지를 뜻합니다.


In [4]:
def attach_embeddings(frame: pd.DataFrame) -> pd.DataFrame:
    lookup = study_rows[["image_id", "origin_embedding"]]
    merged = frame.merge(lookup, on="image_id", how="left", validate="one_to_one")
    if merged["origin_embedding"].isna().any():
        raise ValueError("protocol row에 origin embedding이 없습니다.")
    return merged

def protocol_arrays(protocol):
    gallery = attach_embeddings(protocol.gallery)
    templates = aggregate_templates(
        gallery,
        "mean",
        identity_col="identity_id",
        image_col="image_id",
        embedding_col="origin_embedding",
    )
    query_parts = []
    for probe_type, frame in (
        ("registered", protocol.registered_probes),
        ("known_unknown", protocol.known_unknown_probes),
        ("unknown_unknown", protocol.unknown_unknown_probes),
    ):
        if frame.empty:
            continue
        part = attach_embeddings(frame)
        part["probe_type"] = probe_type
        query_parts.append(part)
    queries = pd.concat(query_parts, ignore_index=True)
    return {
        "query_ids": queries["image_id"].astype(str).to_numpy(),
        "query_identity_ids": queries["identity_id"].astype(str).to_numpy(),
        "queries": np.stack(queries["origin_embedding"].to_numpy()).astype(np.float32),
        "gallery_ids": templates["identity_id"].astype(str).to_numpy(),
        "gallery_identity_ids": templates["identity_id"].astype(str).to_numpy(),
        "gallery": np.stack(templates["embedding"].to_numpy()).astype(np.float32),
    }

def compress_matrix(family: str, compressor, matrix: np.ndarray) -> np.ndarray:
    if family == "pca":
        return compressor.transform(matrix)
    if family == "pq":
        return compressor.transform_profile(matrix).vectors
    raise ValueError(f"unknown compression family: {family}")

def choose_profile_threshold(comparison: pd.DataFrame, score_column: str, correct_column: str) -> float:
    threshold = float(choose_threshold(
        comparison[score_column].to_numpy(dtype=float),
        comparison["is_mated"].to_numpy(dtype=bool),
        comparison[correct_column].to_numpy(dtype=bool),
        TARGET_FPIR,
    ))
    if not np.isfinite(threshold):
        raise ValueError(
            "Threshold calibration returned a non-finite value. "
            "Increase DATA_FRACTION or calibration identities and re-run data preparation."
        )
    return threshold

def profile_storage_metadata(family: str, profile_result) -> dict[str, object]:
    metadata_key = "storage_bytes_per_vector" if family == "pca" else "code_bytes"
    if metadata_key not in profile_result.metadata:
        raise ValueError(f"{family} result is missing metadata[{metadata_key!r}].")
    storage_bytes = int(profile_result.metadata[metadata_key])
    if storage_bytes <= 0:
        raise ValueError(f"{family} storage bytes must be positive, got {storage_bytes}.")
    if family == "pq":
        codebook_bytes = int(profile_result.metadata.get("codebook_bytes", 0))
        codebook_source = str(profile_result.metadata.get("codebook_bytes_source", ""))
        if codebook_bytes <= 0 or not codebook_source:
            raise ValueError("PQ result must report positive codebook metadata.")
    else:
        codebook_bytes = 0
        codebook_source = "not_applicable"
    return {
        "storage_bytes_per_embedding": storage_bytes,
        "codebook_bytes": codebook_bytes,
        "codebook_bytes_source": codebook_source,
    }

def summarize_results(paired: pd.DataFrame, retrieval: pd.DataFrame) -> pd.DataFrame:
    paired_summary = (
        paired.groupby(["compression_family", "compression_profile"], sort=True)
        .agg(
            sample_count=("sample_id", "size"),
            mean_angular_error_rad=("angular_error_rad", "mean"),
            p95_angular_error_rad=("angular_error_rad", lambda values: values.quantile(0.95)),
            mean_reconstruction_mse=("reconstruction_mse", "mean"),
            mean_cosine_to_origin=("cosine_to_origin", "mean"),
            storage_bytes_per_embedding=("storage_bytes_per_embedding", "first"),
            codebook_bytes=("codebook_bytes", "first"),
            codebook_bytes_source=("codebook_bytes_source", "first"),
        )
        .reset_index()
    )
    records = []
    for keys, group in retrieval.groupby(
        ["compression_family", "compression_profile", "threshold_policy"],
        sort=True,
    ):
        family, profile, policy = keys
        mated = group["is_mated"].astype(bool)
        accepted = group["compressed_accepted"].astype(bool)
        correct = group["compressed_rank1_correct"].astype(bool)
        records.append({
            "compression_family": family,
            "compression_profile": profile,
            "threshold_policy": policy,
            "query_count": int(len(group)),
            "mated_count": int(mated.sum()),
            "non_mated_count": int((~mated).sum()),
            "dir_rank1": float((accepted & correct & mated).sum() / mated.sum()) if mated.any() else np.nan,
            "fpir": float((accepted & ~mated).sum() / (~mated).sum()) if (~mated).any() else np.nan,
            "agreement_with_origin": float(group["agreement_with_origin"].mean()),
            "threshold_crossing_rate": float(group["threshold_crossing"].mean()),
        })
    retrieval_summary = pd.DataFrame.from_records(records)
    return retrieval_summary.merge(
        paired_summary,
        on=["compression_family", "compression_profile"],
        how="left",
        validate="many_to_one",
    )


In [5]:
paired_table = None
retrieval_table = None
summary_table = None
if EXECUTE_STAGE:
    training_rows = study_rows.loc[
        study_rows["protocol_role"].astype(str).eq("training")
    ]
    development_rows = training_rows.loc[training_rows["split"].eq("development")]
    official_rows = study_rows.loc[
        ~study_rows["protocol_role"].astype(str).eq("training")
    ]
    development_matrix = np.stack(
        development_rows["origin_embedding"].to_numpy()
    ).astype(np.float32)
    official_matrix = np.stack(
        official_rows["origin_embedding"].to_numpy()
    ).astype(np.float32)
    if len(development_matrix) < max(PCA_DIMENSIONS):
        raise ValueError("PCA-384 fit을 위해 SurvFace development 표본 비율을 늘리십시오.")

    bundle = load_survface_compressor_bundle(
        run,
        pca_dimensions=PCA_DIMENSIONS,
        pq_settings=PQ_SETTINGS,
    )
    if bundle.fit_count != len(development_matrix):
        raise ValueError("frozen compressor fit_count와 현재 development coverage가 다릅니다.")
    pca_models = bundle.pcas
    compressors = [
        ("pca", profile, compressor)
        for profile, compressor in pca_models.items()
    ]
    for m, nbits in PQ_SETTINGS:
        profile = pq_profile_name(m, nbits)
        compressors.append(("pq", profile, bundle.pqs[profile]))

    calibration_identity_sizes = (
        training_manifest.loc[training_manifest["split"].eq("calibration")]
        .groupby("identity_id")["image_id"]
        .nunique()
    )
    eligible_calibration = int((calibration_identity_sizes > 1).sum())
    calibration_gallery_count = min(
        CALIBRATION_GALLERY_IDENTITIES,
        max(1, eligible_calibration),
    )
    calibration_protocol = build_calibration_protocol(
        training_manifest,
        split_name="calibration",
        gallery_identity_count=calibration_gallery_count,
        enrollment_count=1,
        seed=SEED,
    )
    official_protocol = build_survface_official_protocol(official_manifest)
    calibration_arrays = protocol_arrays(calibration_protocol)
    official_arrays = protocol_arrays(official_protocol)

    paired_frames = []
    retrieval_frames = []
    profile_storage = {}
    work_per_profile = 2 * (
        len(calibration_arrays["queries"]) + len(official_arrays["queries"])
    )
    retrieval_work_total = len(compressors) * work_per_profile
    report = PROGRESS.callback(key_prefix=f"{run.run_id}:")
    for profile_index, (family, profile, compressor) in enumerate(compressors):
        profile_result = compressor.transform_profile(official_matrix)
        storage_metadata = profile_storage_metadata(family, profile_result)
        profile_storage[profile] = {
            "compression_family": family, **storage_metadata
        }
        paired = paired_embedding_metrics(
            official_matrix,
            profile_result.vectors,
            reconstructed_embeddings=profile_result.reconstructed_vectors,
            sample_ids=official_rows["image_id"].astype(str).to_numpy(),
            compression_family=family,
            compression_profile=profile,
        )
        paired.insert(0, "dataset", "survface")
        paired.insert(1, "model_name", MODEL_NAME)
        for column, value in storage_metadata.items():
            paired[column] = value
        paired_frames.append(paired)

        calibration_comparison = compare_cosine_retrieval(
            calibration_arrays["queries"],
            calibration_arrays["gallery"],
            compress_matrix(family, compressor, calibration_arrays["queries"]),
            compress_matrix(family, compressor, calibration_arrays["gallery"]),
            query_ids=calibration_arrays["query_ids"],
            gallery_ids=calibration_arrays["gallery_ids"],
            query_identity_ids=calibration_arrays["query_identity_ids"],
            gallery_identity_ids=calibration_arrays["gallery_identity_ids"],
            compression_family=family,
            compression_profile=profile,
            top_k=min(TOP_K, len(calibration_arrays["gallery"])),
            progress=report,
            progress_message="SurvFace Step 1 compression retrieval",
            progress_offset=profile_index * work_per_profile,
            progress_total=retrieval_work_total,
            progress_details={"profile": profile, "split": "training_calibration"},
        )
        origin_threshold = choose_profile_threshold(
            calibration_comparison,
            "origin_top1_score",
            "origin_rank1_correct",
        )
        compressed_threshold = choose_profile_threshold(
            calibration_comparison,
            "compressed_top1_score",
            "compressed_rank1_correct",
        )

        compressed_official_queries = compress_matrix(
            family, compressor, official_arrays["queries"]
        )
        compressed_official_gallery = compress_matrix(
            family, compressor, official_arrays["gallery"]
        )
        official_comparison = compare_cosine_retrieval(
            official_arrays["queries"],
            official_arrays["gallery"],
            compressed_official_queries,
            compressed_official_gallery,
            query_ids=official_arrays["query_ids"],
            gallery_ids=official_arrays["gallery_ids"],
            query_identity_ids=official_arrays["query_identity_ids"],
            gallery_identity_ids=official_arrays["gallery_identity_ids"],
            compression_family=family,
            compression_profile=profile,
            top_k=min(TOP_K, len(official_arrays["gallery"])),
            progress=report,
            progress_message="SurvFace Step 1 compression retrieval",
            progress_offset=(
                profile_index * work_per_profile
                + 2 * len(calibration_arrays["queries"])
            ),
            progress_total=retrieval_work_total,
            progress_details={"profile": profile, "split": "official_test"},
        )
        for threshold_policy, operating_compressed_threshold in (
            ("frozen_origin", origin_threshold),
            ("recalibrated_compressed", compressed_threshold),
        ):
            compared = apply_retrieval_thresholds(
                official_comparison,
                origin_threshold=origin_threshold,
                compressed_threshold=operating_compressed_threshold,
            )
            compared.insert(0, "dataset", "survface")
            compared.insert(1, "model_name", MODEL_NAME)
            compared["threshold_policy"] = threshold_policy
            compared["threshold_source_split"] = "training_calibration"
            compared["evaluation_split"] = "official_test"
            for column, value in storage_metadata.items():
                compared[column] = value
            retrieval_frames.append(compared)

    paired_table = pd.concat(paired_frames, ignore_index=True)
    retrieval_table = pd.concat(retrieval_frames, ignore_index=True)
    if paired_table["origin_fallback_used"].astype(bool).any():
        raise RuntimeError("paired metric에 origin fallback 사용 행이 있습니다.")
    if retrieval_table["origin_fallback_used"].astype(bool).any():
        raise RuntimeError("retrieval 비교에 origin fallback 사용 행이 있습니다.")
    summary_table = summarize_results(paired_table, retrieval_table)
    display(summary_table)
else:
    print("EXECUTE_STAGE=False: 압축 fit과 fallback-free 평가를 실행하지 않았습니다.")


[16:43:43] SurvFace Step 1 characterization | SurvFace Step 1 compression retrieval | elapsed=13m 15s | progress=10% processed=453484 total=4534840 rate=570.45/s eta=1h 59m 15s profile=pca_384 split=official_test representation=compressed
[16:45:26] SurvFace Step 1 characterization | SurvFace Step 1 compression retrieval | elapsed=14m 57s | progress=20% processed=906968 total=4534840 rate=1010.72/s eta=59m 49s profile=pca_256 split=official_test representation=compressed
[16:47:06] SurvFace Step 1 characterization | SurvFace Step 1 compression retrieval | elapsed=16m 38s | progress=30% processed=1360452 total=4534840 rate=1363.29/s eta=38m 48s profile=pca_128 split=official_test representation=compressed
[16:48:45] SurvFace Step 1 characterization | SurvFace Step 1 compression retrieval | elapsed=18m 17s | progress=40% processed=1813936 total=4534840 rate=1653.77/s eta=27m 25s profile=pca_64 split=official_test representation=compressed
[16:50:23] SurvFace Step 1 characterization | Sur

,compression_family,compression_profile,threshold_policy,query_count,mated_count,non_mated_count,dir_rank1,fpir,agreement_with_origin,threshold_crossing_rate,sample_count,mean_angular_error_rad,p95_angular_error_rad,mean_reconstruction_mse,mean_cosine_to_origin,storage_bytes_per_embedding,codebook_bytes,codebook_bytes_source
0,pca,pca_128,frozen_origin,182159,60423,121736,0.005296,0.050642,0.364385,0.474646,242453,0.317079,0.463718,1.978742e-04,0.947644,512,0,not_applicable
1,pca,pca_128,recalibrated_compressed,182159,60423,121736,0.022061,0.558561,0.364385,0.269528,242453,0.317079,0.463718,1.978742e-04,0.947644,512,0,not_applicable
2,pca,pca_256,frozen_origin,182159,60423,121736,0.003459,0.027157,0.417328,0.498158,242453,0.071419,0.104083,1.053494e-05,0.997299,1024,0,not_applicable
3,pca,pca_256,recalibrated_compressed,182159,60423,121736,0.024792,0.549903,0.417328,0.241602,242453,0.071419,0.104083,1.053494e-05,0.997299,1024,0,not_applicable
4,pca,pca_32,frozen_origin,182159,60423,121736,0.011916,0.531618,0.190290,0.370297,242453,0.611777,0.883293,6.540861e-04,0.811237,128,0,not_applicable
5,pca,pca_32,recalibrated_compressed,182159,60423,121736,0.012561,0.602270,0.190290,0.370292,242453,0.611777,0.883293,6.540861e-04,0.811237,128,0,not_applicable
6,pca,pca_384,frozen_origin,182159,60423,121736,0.003376,0.026467,0.419326,0.498910,242453,0.008291,0.010537,1.375184e-07,0.999965,1536,0,not_applicable
7,pca,pca_384,recalibrated_compressed,182159,60423,121736,0.024924,0.548835,0.419326,0.241179,242453,0.008291,0.010537,1.375184e-07,0.999965,1536,0,not_applicable
8,pca,pca_64,frozen_origin,182159,60423,121736,0.008010,0.134192,0.288852,0.414995,242453,0.475901,0.693954,4.217021e-04,0.883815,256,0,not_applicable
9,pca,pca_64,recalibrated_compressed,182159,60423,121736,0.017874,0.576485,0.288852,0.315247,242453,0.475901,0.693954,4.217021e-04,0.883815,256,0,not_applicable


## Export

출력은 완료 extraction run을 수정하지 않고 별도 `results/step1/survface/<run_id>/<scope>/`에 생성합니다. 같은 경로가 존재하면 덮어쓰지 않습니다.


In [6]:
export_result = {"status": "not_executed"}
if EXECUTE_STAGE:
    scope_tag = f"{MODE}_p{DATA_FRACTION:.4f}_s{SEED}_{MODEL_NAME}"
    destination = RESULT_ROOT / run.run_id / scope_tag
    export_result = {
        "status": "computed_not_written",
        "destination": str(destination),
        "paired_rows": int(len(paired_table)),
        "retrieval_rows": int(len(retrieval_table)),
    }
    if WRITE_OUTPUTS:
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.mkdir(exist_ok=False)
        paired_path = destination / "paired_embedding_metrics.csv"
        retrieval_path = destination / "retrieval_comparison.csv"
        summary_path = destination / "compression_summary.csv"
        paired_table.to_csv(paired_path, index=False, encoding="utf-8", lineterminator="\n")
        retrieval_table.to_csv(retrieval_path, index=False, encoding="utf-8", lineterminator="\n")
        summary_table.to_csv(summary_path, index=False, encoding="utf-8", lineterminator="\n")
        files = {
            path.name: {"sha256": sha256_file(path), "bytes": path.stat().st_size}
            for path in (paired_path, retrieval_path, summary_path)
        }
        result_manifest = {
            "schema_version": 1,
            "dataset": "survface",
            "model_name": MODEL_NAME,
            "scope": EXPERIMENT_SCOPE.as_dict(),
            "compression": {
                "pca_dimensions": list(PCA_DIMENSIONS),
                "pq_settings": [
                    {"m": m, "nbits": nbits, "source_dimension": PQ_SOURCE_DIMENSION}
                    for m, nbits in PQ_SETTINGS
                ],
                "profile_storage": profile_storage,
                "pca_to_pq": False,
            },
            "evaluation": {
                "target_fpir": TARGET_FPIR,
                "threshold_policies": ["frozen_origin", "recalibrated_compressed"],
                "origin_fallback": False,
                "fit_source": "training_development",
                "threshold_source": "training_calibration",
                "official_test_role": "evaluation_only",
            },
            "source": {
                "extraction_run_id": run.run_id,
                "config": str(CONFIG_PATH.relative_to(PROJECT_ROOT)),
                "config_sha256": sha256_file(CONFIG_PATH),
                "training_manifest": str(TRAINING_MANIFEST_PATH.relative_to(PROJECT_ROOT)),
                "training_manifest_sha256": sha256_file(TRAINING_MANIFEST_PATH),
                "official_manifest": str(OFFICIAL_MANIFEST_PATH.relative_to(PROJECT_ROOT)),
                "official_manifest_sha256": sha256_file(OFFICIAL_MANIFEST_PATH),
            },
            "required_columns": {
                "paired": list(PAIRED_EMBEDDING_COLUMNS),
                "retrieval": list(RETRIEVAL_COMPARISON_COLUMNS),
            },
            "files": files,
        }
        manifest_path = destination / "result_manifest.json"
        manifest_path.write_text(
            json.dumps(result_manifest, ensure_ascii=False, indent=2, sort_keys=True) + "\n",
            encoding="utf-8",
        )
        export_result = {
            "status": "written",
            "destination": str(destination),
            "manifest": str(manifest_path),
            "files": sorted(files),
        }
export_result


{'status': 'written',
 'destination': 'C:\\ronbun\\results\\step1\\survface\\20260729-R001-3c19d8dc\\real_p1.0000_s42_arcface',
 'manifest': 'C:\\ronbun\\results\\step1\\survface\\20260729-R001-3c19d8dc\\real_p1.0000_s42_arcface\\result_manifest.json',
 'files': ['compression_summary.csv',
  'paired_embedding_metrics.csv',
  'retrieval_comparison.csv']}

## Final check

- `training_summary.json`에서 official test가 fit split에 포함되지 않았는지 확인합니다.
- `MODE=real`, `DATA_FRACTION=1.0`과 전체 공식 protocol이 아니면 최종 SurvFace 결과로 사용하지 않습니다.
- `result_manifest.json`의 `pca_to_pq=false`, `origin_fallback=false`를 확인합니다.
- 낮은 FPIR을 주장하기 전에 선택된 unmated probe 수가 충분한지 확인합니다.
- 다음으로 공통 결과 노트북에서 LFW와 함께 집계합니다.
